# 05 — Hyperparameter Tuning

This notebook tunes the strongest baseline fraud models identified in the previous stage.

Primary model-selection metric:
- PR-AUC

Secondary metrics:
- ROC-AUC
- Recall
- Precision
- F1-score

The validation period remains month 6, while month 7 is kept untouched for final evaluation.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    r"D:\bank-account-fraud-decision-system"
)

data_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "fraud_modeling_dataset.parquet"
)

df = pd.read_parquet(data_path)

train_df = df[
    df["month"].between(0, 5)
].copy()

val_df = df[
    df["month"] == 6
].copy()

test_df = df[
    df["month"] == 7
].copy()

target = "fraud_bool"

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_val = val_df.drop(columns=[target])
y_val = val_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

In [3]:
cat_features = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os"
]

X_train_cb = X_train.copy()
X_val_cb = X_val.copy()

for col in cat_features:
    X_train_cb[col] = (
        X_train_cb[col]
        .fillna("Missing")
        .astype(str)
    )

    X_val_cb[col] = (
        X_val_cb[col]
        .fillna("Missing")
        .astype(str)
    )

In [4]:
from catboost import CatBoostClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

catboost_trials = [
    {
        "depth": 5,
        "learning_rate": 0.05,
        "iterations": 600
    },
    {
        "depth": 6,
        "learning_rate": 0.05,
        "iterations": 800
    },
    {
        "depth": 7,
        "learning_rate": 0.03,
        "iterations": 800
    },
    {
        "depth": 8,
        "learning_rate": 0.03,
        "iterations": 1000
    }
]

In [5]:
catboost_results = []

for i, params in enumerate(
    catboost_trials,
    start=1
):
    print(f"\nTrial {i}: {params}")

    model = CatBoostClassifier(
        **params,
        loss_function="Logloss",
        eval_metric="PRAUC",
        auto_class_weights="Balanced",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train_cb,
        y_train,
        cat_features=cat_features,
        eval_set=(X_val_cb, y_val),
        use_best_model=True
    )

    y_val_proba = model.predict_proba(
        X_val_cb
    )[:, 1]

    pr_auc = average_precision_score(
        y_val,
        y_val_proba
    )

    roc_auc = roc_auc_score(
        y_val,
        y_val_proba
    )

    catboost_results.append({
        "trial": i,
        **params,
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "best_iteration": model.get_best_iteration()
    })

catboost_results_df = pd.DataFrame(
    catboost_results
).sort_values(
    "PR_AUC",
    ascending=False
)

display(catboost_results_df)


Trial 1: {'depth': 5, 'learning_rate': 0.05, 'iterations': 600}

Trial 2: {'depth': 6, 'learning_rate': 0.05, 'iterations': 800}

Trial 3: {'depth': 7, 'learning_rate': 0.03, 'iterations': 800}

Trial 4: {'depth': 8, 'learning_rate': 0.03, 'iterations': 1000}


,trial,depth,learning_rate,iterations,PR_AUC,ROC_AUC,best_iteration
0,1,5,0.05,600,0.185945,0.894195,577
1,2,6,0.05,800,0.180778,0.891688,516
2,3,7,0.03,800,0.179880,0.890830,588
3,4,8,0.03,1000,0.177950,0.887914,619


## CatBoost Tuning — Round 2

The first tuning round showed that a shallower CatBoost model performed best. A narrower second search is therefore conducted around depth 4–6 with moderate learning rates and iteration counts.

In [6]:
catboost_trials_round2 = [
    {
        "depth": 4,
        "learning_rate": 0.05,
        "iterations": 700
    },
    {
        "depth": 5,
        "learning_rate": 0.04,
        "iterations": 800
    },
    {
        "depth": 5,
        "learning_rate": 0.05,
        "iterations": 700
    },
    {
        "depth": 5,
        "learning_rate": 0.06,
        "iterations": 600
    },
    {
        "depth": 6,
        "learning_rate": 0.04,
        "iterations": 700
    }
]

In [7]:
catboost_results_round2 = []

for i, params in enumerate(
    catboost_trials_round2,
    start=1
):
    print(f"\nRound 2 - Trial {i}: {params}")

    model = CatBoostClassifier(
        **params,
        loss_function="Logloss",
        eval_metric="PRAUC",
        auto_class_weights="Balanced",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train_cb,
        y_train,
        cat_features=cat_features,
        eval_set=(X_val_cb, y_val),
        use_best_model=True
    )

    y_val_proba = model.predict_proba(
        X_val_cb
    )[:, 1]

    pr_auc = average_precision_score(
        y_val,
        y_val_proba
    )

    roc_auc = roc_auc_score(
        y_val,
        y_val_proba
    )

    catboost_results_round2.append({
        "trial": i,
        **params,
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "best_iteration": model.get_best_iteration()
    })

catboost_results_round2_df = pd.DataFrame(
    catboost_results_round2
).sort_values(
    "PR_AUC",
    ascending=False
)

display(catboost_results_round2_df)


Round 2 - Trial 1: {'depth': 4, 'learning_rate': 0.05, 'iterations': 700}

Round 2 - Trial 2: {'depth': 5, 'learning_rate': 0.04, 'iterations': 800}

Round 2 - Trial 3: {'depth': 5, 'learning_rate': 0.05, 'iterations': 700}

Round 2 - Trial 4: {'depth': 5, 'learning_rate': 0.06, 'iterations': 600}

Round 2 - Trial 5: {'depth': 6, 'learning_rate': 0.04, 'iterations': 700}


,trial,depth,learning_rate,iterations,PR_AUC,ROC_AUC,best_iteration
2,3,5,0.05,700,0.186078,0.894221,631
0,1,4,0.05,700,0.184674,0.894529,658
4,5,6,0.04,700,0.183449,0.892911,613
3,4,5,0.06,600,0.183007,0.893371,443
1,2,5,0.04,800,0.182456,0.893589,799


### CatBoost Tuning Conclusion

The second tuning round confirmed that a moderately shallow CatBoost model performs best.

The selected configuration uses:

- Depth: 5
- Learning rate: 0.05
- Maximum iterations: 700
- Best validation iteration: approximately 631

This configuration achieved a PR-AUC of 0.1861 and ROC-AUC of 0.8942.

Additional CatBoost tuning was stopped because performance improvements had become marginal, reducing the risk of overfitting model-selection decisions to the validation month.

In [16]:
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

In [17]:
xgb_trials = [
    {
        "max_depth": 4,
        "learning_rate": 0.05,
        "n_estimators": 600
    },
    {
        "max_depth": 5,
        "learning_rate": 0.05,
        "n_estimators": 600
    },
    {
        "max_depth": 6,
        "learning_rate": 0.05,
        "n_estimators": 600
    },
    {
        "max_depth": 5,
        "learning_rate": 0.03,
        "n_estimators": 800
    }
]

In [18]:
xgb_results = []

for i, params in enumerate(xgb_trials, start=1):

    print(f"\nXGBoost Trial {i}: {params}")

    model = XGBClassifier(
        **params,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_processed,
        y_train
    )

    y_val_proba = model.predict_proba(
        X_val_processed
    )[:, 1]

    pr_auc = average_precision_score(
        y_val,
        y_val_proba
    )

    roc_auc = roc_auc_score(
        y_val,
        y_val_proba
    )

    xgb_results.append({
        "trial": i,
        **params,
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc
    })

xgb_results_df = (
    pd.DataFrame(xgb_results)
    .sort_values(
        "PR_AUC",
        ascending=False
    )
)

display(xgb_results_df)


XGBoost Trial 1: {'max_depth': 4, 'learning_rate': 0.05, 'n_estimators': 600}

XGBoost Trial 2: {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 600}

XGBoost Trial 3: {'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 600}

XGBoost Trial 4: {'max_depth': 5, 'learning_rate': 0.03, 'n_estimators': 800}


,trial,max_depth,learning_rate,n_estimators,PR_AUC,ROC_AUC
0,1,4,0.05,600,0.182008,0.895155
1,2,5,0.05,600,0.180774,0.892175
3,4,5,0.03,800,0.179649,0.893665
2,3,6,0.05,600,0.173576,0.887792


### XGBoost Tuning Conclusion

The best XGBoost configuration used a shallower tree depth of 4, a learning rate of 0.05, and 600 estimators.

This model achieved a PR-AUC of 0.1820 and ROC-AUC of 0.8952. Although XGBoost produced the highest ROC-AUC among the tuned models so far, its PR-AUC remained below the tuned CatBoost model.

Because PR-AUC is the primary selection metric for this highly imbalanced fraud-detection problem, CatBoost remains the leading candidate.

## LightGBM Hyperparameter Tuning

LightGBM is tuned using a compact parameter search focused on tree complexity, learning rate, and number of estimators.

PR-AUC remains the primary selection metric because fraud prevalence is very low.

In [20]:
from lightgbm import LGBMClassifier

lgbm_trials = [
    {
        "num_leaves": 15,
        "learning_rate": 0.05,
        "n_estimators": 600
    },
    {
        "num_leaves": 31,
        "learning_rate": 0.05,
        "n_estimators": 600
    },
    {
        "num_leaves": 63,
        "learning_rate": 0.03,
        "n_estimators": 800
    },
    {
        "num_leaves": 31,
        "learning_rate": 0.03,
        "n_estimators": 800
    }
]

In [21]:
lgbm_results = []

for i, params in enumerate(
    lgbm_trials,
    start=1
):
    print(f"\nLightGBM Trial {i}: {params}")

    model = LGBMClassifier(
        **params,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train_processed,
        y_train
    )

    y_val_proba = model.predict_proba(
        X_val_processed
    )[:, 1]

    pr_auc = average_precision_score(
        y_val,
        y_val_proba
    )

    roc_auc = roc_auc_score(
        y_val,
        y_val_proba
    )

    lgbm_results.append({
        "trial": i,
        **params,
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc
    })

lgbm_results_df = (
    pd.DataFrame(lgbm_results)
    .sort_values(
        "PR_AUC",
        ascending=False
    )
)

display(lgbm_results_df)


LightGBM Trial 1: {'num_leaves': 15, 'learning_rate': 0.05, 'n_estimators': 600}

LightGBM Trial 2: {'num_leaves': 31, 'learning_rate': 0.05, 'n_estimators': 600}

LightGBM Trial 3: {'num_leaves': 63, 'learning_rate': 0.03, 'n_estimators': 800}

LightGBM Trial 4: {'num_leaves': 31, 'learning_rate': 0.03, 'n_estimators': 800}


,trial,num_leaves,learning_rate,n_estimators,PR_AUC,ROC_AUC
3,4,31,0.03,800,0.175833,0.891346
1,2,31,0.05,600,0.169937,0.887006
0,1,15,0.05,600,0.169884,0.891063
2,3,63,0.03,800,0.166219,0.885171


### LightGBM Tuning Conclusion

The best LightGBM configuration used 31 leaves, a learning rate of 0.03, and 800 estimators.

It achieved a PR-AUC of 0.1758 and ROC-AUC of 0.8913 on the validation period. Although this improved on the initial LightGBM baseline, it remained below the tuned CatBoost and XGBoost models on the primary PR-AUC metric.

LightGBM is therefore retained as a strong benchmark but is not selected as the primary model.

## Logistic Regression Hyperparameter Tuning

Logistic Regression is tuned as a lightweight linear benchmark.

The search focuses on:
- Regularization strength (`C`)
- L1 vs L2 penalty

PR-AUC remains the primary metric.

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

logreg_trials = [
    {
        "penalty": "l2",
        "C": 0.01,
        "solver": "liblinear"
    },
    {
        "penalty": "l2",
        "C": 0.1,
        "solver": "liblinear"
    },
    {
        "penalty": "l2",
        "C": 1.0,
        "solver": "liblinear"
    },
    {
        "penalty": "l1",
        "C": 0.1,
        "solver": "liblinear"
    },
    {
        "penalty": "l1",
        "C": 1.0,
        "solver": "liblinear"
    }
]

## Final Tuned Model Comparison

The tuned models were compared using month 6 as the validation period. PR-AUC was prioritized because fraud represents only a small fraction of applications and the project focuses on correctly ranking high-risk applications.

CatBoost achieved the highest validation PR-AUC and is selected as the primary model for the next stage. XGBoost remains the strongest alternative model and achieved the highest ROC-AUC.

The month 7 test set has remained untouched throughout model development and hyperparameter selection.